# 1) Load Data
Loading raw game data collected from the Steam API. Data contains nested JSON fields that need to be flattened before analysis.

In [357]:
import pandas as pd
import ast

In [358]:
data = pd.read_csv("../data/raw/games_raw.csv")

print(data.shape)
print(data.isnull().sum())
display(data.dtypes)

(22171, 11)
steam_appid            0
name                   1
is_free                0
developers            28
publishers           101
price_overview      3441
genres                17
categories             3
release_date           0
recommendations    18209
metacritic         21387
dtype: int64


steam_appid        int64
name                 str
is_free             bool
developers           str
publishers           str
price_overview       str
genres               str
categories           str
release_date         str
recommendations      str
metacritic           str
dtype: object

In [359]:
data.head()

,steam_appid,name,is_free,developers,publishers,price_overview,genres,categories,release_date,recommendations,metacritic
0,10,Counter-Strike,False,['Valve'],['Valve'],"{'currency': 'USD', 'initial': 579, 'final': 5...","[{'id': '1', 'description': 'Action'}]","[{'id': 1, 'description': 'Multi-player'}, {'i...","{'coming_soon': False, 'date': '1 Nov, 2000'}",{'total': 168140},"{'score': 88, 'url': 'https://www.metacritic.c..."
1,20,Team Fortress Classic,False,['Valve'],['Valve'],"{'currency': 'USD', 'initial': 499, 'final': 4...","[{'id': '1', 'description': 'Action'}]","[{'id': 1, 'description': 'Multi-player'}, {'i...","{'coming_soon': False, 'date': 'Apr 1, 1999'}",{'total': 6947},NaN
2,30,Day of Defeat,False,['Valve'],['Valve'],"{'currency': 'USD', 'initial': 499, 'final': 4...","[{'id': '1', 'description': 'Action'}]","[{'id': 1, 'description': 'Multi-player'}, {'i...","{'coming_soon': False, 'date': 'May 1, 2003'}",{'total': 4426},"{'score': 79, 'url': 'https://www.metacritic.c..."
3,40,Deathmatch Classic,False,['Valve'],['Valve'],"{'currency': 'USD', 'initial': 499, 'final': 4...","[{'id': '1', 'description': 'Action'}]","[{'id': 1, 'description': 'Multi-player'}, {'i...","{'coming_soon': False, 'date': 'Jun 1, 2001'}",{'total': 2420},NaN
4,50,Half-Life: Opposing Force,False,['Gearbox Software'],['Valve'],"{'currency': 'USD', 'initial': 499, 'final': 4...","[{'id': '1', 'description': 'Action'}]","[{'id': 2, 'description': 'Single-player'}, {'...","{'coming_soon': False, 'date': 'Nov 1, 1999'}",{'total': 24704},NaN


# 2) Pre-processing Steps
Extracting relevant values from nested dictionary columns, converting price to USD, standardizing date format, and dropping columns that are non-relavant to the goal.

## 2.1) Parse Nested Columns

In [360]:
data["developers"] = data["developers"].apply(lambda x: ast.literal_eval(x) if not pd.isna(x) else None)

data["publishers"] = data["publishers"].apply(lambda x: ast.literal_eval(x) if not pd.isna(x) else None)

data["recommendations"] = data["recommendations"].apply(lambda x: ast.literal_eval(x) if not pd.isna(x) else None)
data["recommendations"] = data["recommendations"].apply(lambda x: x.get("total") if not pd.isna(x) else None)

data["genres"] = data["genres"].apply(lambda x: ast.literal_eval(x) if not pd.isna(x) else None)
data["genres"] = data["genres"].apply(lambda x: [dev.get("description") for dev in x] if x is not None else None)

data["metacritic"] = data["metacritic"].apply(lambda x: ast.literal_eval(x) if not pd.isna(x) else None)
data["metacritic"] = data["metacritic"].apply(lambda x: x.get("score") if not pd.isna(x) else None)

## 2.2) Clean Price

In [361]:
data["price_overview"] = data["price_overview"].apply(lambda x: ast.literal_eval(x) if not pd.isna(x) else None)
data["price_overview"] = (data["price_overview"].apply(lambda x: 0 if pd.isna(x) else x.get("final") if x.get("currency") == "USD" else None) / 100)
data = data.rename(columns={"price_overview": "price_usd"})

## 2.3) Format Dates

In [362]:
data["release_date"] = data["release_date"].apply(lambda x: ast.literal_eval(x) if not pd.isna(x) else None)
data["release_date"] = data["release_date"].apply(lambda x: x.get("date") if not pd.isna(x) else None)
data["release_date"] = pd.to_datetime(data["release_date"], format="mixed", errors="coerce")

## 2.4) Configure Columns and Rows

Rows missing values in developers, publishers, price_usd, genres, and release_date were dropped as these fields are central to the goal, which reduced the data from 22,171 to 20,516 rows. Metacritic scores were retained as null rather than dropped as that would reduce the size of the data significantly and impact future analysis. Dropping these would introduce significant survivorship bias toward larger, more established titles. 17,065 games have no recommendation data, which was confirmed against Steam store pages. These are retained in the dataset but will be excluded from analyses requiring review metrics.

In [363]:
data = data.drop(columns=["is_free", "categories"])
data = data.dropna(subset=["developers", "publishers", "price_usd", "genres", "release_date"])

In [364]:
print(data.shape)
print(data.isnull().sum())
display(data.dtypes)

(20516, 9)
steam_appid            0
name                   0
developers             0
publishers             0
price_usd              0
genres                 0
release_date           0
recommendations    17065
metacritic         19827
dtype: int64


steam_appid                 int64
name                          str
developers                 object
publishers                 object
price_usd                 float64
genres                     object
release_date       datetime64[us]
recommendations           float64
metacritic                float64
dtype: object

In [365]:
data.head()

,steam_appid,name,developers,publishers,price_usd,genres,release_date,recommendations,metacritic
0,10,Counter-Strike,[Valve],[Valve],5.79,[Action],2000-11-01,168140.0,88.0
1,20,Team Fortress Classic,[Valve],[Valve],4.99,[Action],1999-04-01,6947.0,NaN
2,30,Day of Defeat,[Valve],[Valve],4.99,[Action],2003-05-01,4426.0,79.0
3,40,Deathmatch Classic,[Valve],[Valve],4.99,[Action],2001-06-01,2420.0,NaN
4,50,Half-Life: Opposing Force,[Gearbox Software],[Valve],4.99,[Action],1999-11-01,24704.0,NaN


# 3. Save Data

In [366]:
data.to_csv("../data/processed/games_clean.csv", index=False)